# Modelo de Detección de Fraude Financiero
## Caso Integrador Final — Programa de Especialización en Credit Scoring con Python

**Escenario:** Una institución financiera requiere fortalecer su sistema de gestión de riesgo
operacional mediante un modelo de ML capaz de identificar transacciones anómalas (fraude).

**Variable objetivo:** `fraud` — binaria (0 = legítima, 1 = fraude), tasa de incidencia ~2%.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve,
                              precision_recall_curve, average_precision_score,
                              f1_score, recall_score)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib

RANDOM_STATE = 42
TEST_SIZE    = 0.20
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.figsize"] = (10, 5)

print("Librerias cargadas correctamente.")

In [ ]:
df = pd.read_csv("../data/base.csv")
print(f"Shape: {df.shape}")
print(f"
Columnas: {df.columns.tolist()}")
df.head()

## Módulo 1: Análisis Exploratorio de Datos (EDA)

> "Un buen modelo no nace en el algoritmo, nace en el EDA." — Instructor del programa.

El EDA es la base diagnóstica que justifica todas las decisiones técnicas posteriores.
Se analiza la calidad de los datos, el desbalance de clases, la distribución de variables
y el comportamiento diferenciado entre transacciones legítimas y fraudulentas.

In [ ]:
print("=== ESTADÍSTICOS DESCRIPTIVOS ===")
display(df.describe(percentiles=[.25, .50, .75, .90, .99]).round(2))

print("
=== VALORES NULOS ===")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.any() else "No se encontraron valores nulos.")
print(f"
Tipos de datos:
{df.dtypes}")

In [ ]:
conteo     = df['fraud'].value_counts()
pct_fraude = df['fraud'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(['Legítima (0)', 'Fraude (1)'], conteo.values,
            color=['steelblue', 'tomato'])
axes[0].set_title('Distribución de la variable objetivo')
axes[0].set_ylabel('Cantidad de transacciones')
for i, v in enumerate(conteo.values):
    axes[0].text(i, v + 50, str(v), ha='center', fontweight='bold')

axes[1].pie(conteo.values, labels=['Legítima', 'Fraude'],
            autopct='%1.2f%%', colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Proporción')

plt.suptitle(f'Desbalance de clases — Tasa de fraude: {pct_fraude:.2f}%',
             fontweight='bold')
plt.tight_layout()
plt.show()

print(f"
→ {conteo[0]:,} legítimas vs {conteo[1]:,} fraudes ({pct_fraude:.2f}%)")
print("→ Estrategia requerida: SMOTE + métricas Recall/F1 (no Accuracy global)")

In [ ]:
num_cols = ['amount', 'client_credit_score', 'transaction_frequency',
            'customer_age', 'annual_income', 'account_balance',
            'num_previous_loans', 'customer_tenure', 'num_dependents',
            'education_level']

fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].set_title(col)
plt.suptitle('Distribuciones de variables numéricas', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.7))
    axes[i].set_title(col)
plt.suptitle('Detección de outliers (boxplots)', fontweight='bold')
plt.tight_layout()
plt.show()
print('→ No se eliminan registros. Outliers reales se conservan mediante Winsorización al p99.')

In [ ]:
corr = df[num_cols + ['fraud']].corr(method='spearman')

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlación de Spearman
'
          '(Se usa Spearman: más robusto ante outliers y relaciones no lineales que Pearson)',
          fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(18, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        subset = df[df['fraud'] == label][col]
        axes[i].hist(subset, bins=25, alpha=0.6, density=True,
                     label=f'fraud={label}', color=color)
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)
plt.suptitle('Distribución por clase: Legítima vs Fraude', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['transaction_type', 'location', 'marital_status', 'housing_type']

fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    tasa = df.groupby(col)['fraud'].mean().sort_values(ascending=False)
    tasa.plot(kind='bar', ax=axes[i], color='tomato', alpha=0.8, edgecolor='white')
    axes[i].set_title(f'Tasa de fraude por {col}')
    axes[i].set_ylabel('Proporción de fraude')
    axes[i].tick_params(axis='x', rotation=30)
    for p in axes[i].patches:
        axes[i].annotate(f'{p.get_height():.3f}',
                         (p.get_x() + p.get_width() / 2., p.get_height()),
                         ha='center', va='bottom', fontsize=9)
plt.suptitle('Tasa de fraude por variable categórica', fontweight='bold')
plt.tight_layout()
plt.show()

## Módulo 2: Preprocesamiento de Datos

**Regla de oro anti data leakage:** La división Train/Test se realiza PRIMERO.
Ninguna transformación (escalado, SMOTE, imputación, Winsorización) se aplica antes
de esta división. Los transformadores se ajustan exclusivamente sobre el conjunto de
entrenamiento (`.fit_transform()`) y luego se aplican al de prueba (`.transform()`).

In [ ]:
df['timestamp']  = pd.to_datetime(df['timestamp'])
df['hora']        = df['timestamp'].dt.hour
df['dia_semana']  = df['timestamp'].dt.dayofweek   # 0=lunes, 6=domingo
df['mes']         = df['timestamp'].dt.month

df = df.drop(columns=['transaction_id', 'timestamp'])

print("Feature engineering completado.")
print(f"Nuevas columnas: hora, dia_semana, mes")
print(f"Shape actualizado: {df.shape}")

In [ ]:
X = df.drop(columns=['fraud'])
y = df['fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]:,} | Fraudes: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Test:  {X_test.shape[0]:,}  | Fraudes: {y_test.sum()}  ({y_test.mean()*100:.2f}%)")
print("
→ stratify=y garantiza la misma proporción de fraude en ambos conjuntos.")

In [ ]:
num_features = ['amount', 'client_credit_score', 'transaction_frequency',
                'customer_age', 'annual_income', 'account_balance',
                'num_previous_loans', 'customer_tenure', 'num_dependents',
                'education_level', 'hora', 'dia_semana', 'mes']

cat_features = ['transaction_type', 'location', 'marital_status', 'housing_type']

print(f"Variables numéricas ({len(num_features)}): {num_features}")
print(f"Variables categóricas ({len(cat_features)}): {cat_features}")
print("
→ education_level ya viene codificada ordinalmente (1-4), se trata como numérica.")

In [ ]:
winsor_limits = {}
for col in ['amount', 'annual_income', 'account_balance']:
    p99 = X_train[col].quantile(0.99)
    winsor_limits[col] = p99
    X_train[col] = X_train[col].clip(upper=p99)
    X_test[col]  = X_test[col].clip(upper=p99)

print("Winsorización al p99 aplicada:")
for col, lim in winsor_limits.items():
    print(f"  {col}: límite superior = {lim:,.2f}")
print("
→ Se conservan todos los registros. Solo se acotan valores extremos.")
print("→ El límite se calcula sobre train y se aplica al test (anti data leakage).")

In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler())
    ]), num_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot',  OneHotEncoder(drop='first', sparse_output=False,
                                  handle_unknown='ignore'))
    ]), cat_features)
])

print("ColumnTransformer definido.")
print("→ StandardScaler incluido para Regresión Logística.")
print("→ RF y XGBoost no lo requieren, pero no afecta su desempeño.")

In [ ]:
X_train_prep = preprocessor.fit_transform(X_train, y_train)
X_test_prep  = preprocessor.transform(X_test)

print(f"X_train_prep shape: {X_train_prep.shape}")
print(f"X_test_prep shape:  {X_test_prep.shape}")
print("
→ .fit_transform() solo en train. .transform() en test.")

## Módulo 3: Estrategia ante Desbalance y Construcción del Modelo

**Estrategia de desbalance: SMOTE (Synthetic Minority Over-sampling Technique)**
SMOTE genera instancias sintéticas de la clase minoritaria (fraude) interpolando entre
muestras reales vecinas. Se aplica ÚNICAMENTE sobre el conjunto de entrenamiento
preprocesado. El test permanece con distribución real (~2% fraude).

Se entrenan tres modelos con StratifiedKFold(5) y RandomizedSearchCV:
- **Regresión Logística** (baseline) — modelo de referencia para parsimonia
- **Random Forest** — ensamble por Bagging
- **XGBoost** — ensamble por Boosting

**Principio de parsimonia:** Si RL obtiene métricas similares a RF/XGBoost (diferencia
F1 ≤ 0.03), se seleccionará RL por su mayor interpretabilidad ante reguladores.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = smote.fit_resample(X_train_prep, y_train)

print(f"Antes de SMOTE  → {X_train_prep.shape[0]:,} muestras | fraudes: {y_train.sum()} ({y_train.mean()*100:.2f}%)")
print(f"Después de SMOTE → {X_train_res.shape[0]:,} muestras | fraudes: {y_train_res.sum()} ({y_train_res.mean()*100:.2f}%)")
print("
→ SMOTE solo aplicado al train. El test mantiene distribución real.")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("StratifiedKFold(5) definido.")
print("→ Preserva la proporción de clases en cada fold durante el ajuste de hiperparámetros.")

In [ ]:
param_grid_rl = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}

search_rl = RandomizedSearchCV(
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    param_grid_rl, n_iter=6, cv=cv,
    scoring='f1', refit=True, random_state=RANDOM_STATE, n_jobs=-1
)
search_rl.fit(X_train_res, y_train_res)
best_rl = search_rl.best_estimator_

print(f"Mejor C para RL: {search_rl.best_params_['C']}")
print(f"F1 CV (train):   {search_rl.best_score_:.4f}")

In [ ]:
param_grid_rf = {
    'n_estimators':   [100, 200, 300],
    'max_depth':      [5, 10, 15, None],
    'min_samples_leaf': [1, 2, 5],
    'class_weight':   ['balanced']
}

search_rf = RandomizedSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_grid_rf, n_iter=20, cv=cv,
    scoring='f1', refit=True, random_state=RANDOM_STATE, n_jobs=-1
)
search_rf.fit(X_train_res, y_train_res)
best_rf = search_rf.best_estimator_

print(f"Mejores hiperparámetros RF: {search_rf.best_params_}")
print(f"F1 CV (train):              {search_rf.best_score_:.4f}")

In [ ]:
param_grid_xgb = {
    'n_estimators':    [100, 200, 300],
    'max_depth':       [3, 5, 7],
    'learning_rate':   [0.01, 0.05, 0.1, 0.2],
    'subsample':       [0.7, 0.8, 1.0],
    'colsample_bytree':[0.7, 0.8, 1.0]
}

search_xgb = RandomizedSearchCV(
    xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss'),
    param_grid_xgb, n_iter=20, cv=cv,
    scoring='f1', refit=True, random_state=RANDOM_STATE, n_jobs=-1
)
search_xgb.fit(X_train_res, y_train_res)
best_xgb = search_xgb.best_estimator_

print(f"Mejores hiperparámetros XGB: {search_xgb.best_params_}")
print(f"F1 CV (train):               {search_xgb.best_score_:.4f}")

In [ ]:
joblib.dump(best_rl,        '../models/logistic_regression.pkl')
joblib.dump(best_rf,        '../models/random_forest.pkl')
joblib.dump(best_xgb,       '../models/xgboost.pkl')
joblib.dump(preprocessor,   '../models/preprocessor.pkl')

print("Modelos guardados en /models/")